In [1]:
#TEST

In [3]:
# ==============================================
# SwipeSense — Cloud (GCS) Notebook v10 (single cell)
# ==============================================
# 1) عبّئ CONFIG أدناه
# 2) شغّل الخلية بالكامل
# 3) تحكّم من الواجهة: Next / Like + السلايدرز

# ---- minimal deps ----
%pip install -q -U transformers torch safetensors huggingface_hub pillow numpy ipywidgets "google-cloud-storage<3,>=2.10.0" "gcsfs>=2024.2.0" "fsspec>=2024.2.0"

# ---- Imports ----
from __future__ import annotations
from pathlib import Path
from dataclasses import dataclass
from collections import deque, defaultdict
from io import BytesIO
from typing import List, Dict, Tuple

import time, json, random, math
import numpy as np
from PIL import Image, ImageFile

import torch
from transformers import CLIPProcessor, CLIPModel

import ipywidgets as W
from IPython.display import display, clear_output

# ---- Device & CLIP model ----
device   = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "openai/clip-vit-base-patch32"  # 512-dim
processor = CLIPProcessor.from_pretrained(MODEL_ID, use_fast=False)
model     = CLIPModel.from_pretrained(MODEL_ID).to(device)
model.eval()
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ---- CONFIG ----
USE_GCS: bool   = True                       # True: read from Google Cloud Storage
GCS_BUCKET: str = "swipe-bucket"             # <<< ضع اسم البكِت
GCS_PREFIX: str = ""                         # مجلد داخل البكِت (ممكن فاضي)
MAX_IMAGES: int = 100                        # عدد الصور المستخدمة
SEED: int       = 123                        # للثبات

# Artifacts & persistence
MIRROR_TO_GCS: bool = False                  # True: ارفع ملفات الكاش/المستخدمين إلى GCS
ARTIFACTS_PREFIX: str = "artifacts/"         # مجلد داخل البكِت لحفظ النسخ
USER_DB_PATH = Path("./session_state/users.json")  # قاعدة تفضيلات المستخدمين

# Feedback mapping (ثابت حسب طلبك)
LIKE_BASE = 0.30
SWIPE_FAST = -0.10
DWELL_THRESHOLD = 3.0
DWELL_BONUS = 0.10  # تُستخدم مع السوايب فقط

# Optional local dirs
LOCAL_DIRS = [Path.cwd() / "images", Path.cwd() / "data", Path.cwd()]

# ---- Utils ----
def l2_normalize(x: np.ndarray, axis=-1, eps=1e-12):
    n = np.linalg.norm(x, axis=axis, keepdims=True)
    return x / np.maximum(n, eps)

# ==============================================
# GCS + Image IO
# ==============================================
from google.cloud import storage
import gcsfs

_gcs_client = None
_gcs_fs = None

def _ensure_gcs():
    global _gcs_client, _gcs_fs
    if _gcs_client is None:
        _gcs_client = storage.Client()   # uses VM/ADC creds
    if _gcs_fs is None:
        _gcs_fs = gcsfs.GCSFileSystem(token='cloud')

def _upload_to_gcs(local_path: str, dst_name: str | None = None):
    if not (USE_GCS and MIRROR_TO_GCS):
        return
    _ensure_gcs()
    dst_name = dst_name or (ARTIFACTS_PREFIX.rstrip('/') + '/' + Path(local_path).name)
    bucket = _gcs_client.bucket(GCS_BUCKET)
    blob = bucket.blob(dst_name)
    blob.upload_from_filename(local_path)

IMG_EXTS = (".jpg", ".jpeg", ".png", ".webp")

def list_gcs_images(bucket: str, prefix: str = "", limit: int | None = None) -> List[str]:
    _ensure_gcs()
    blobs = _gcs_client.list_blobs(bucket, prefix=prefix or None)
    out = []
    for b in blobs:
        name = b.name
        if name.lower().endswith(IMG_EXTS):
            out.append(f"gs://{bucket}/{name}")
            if limit is not None and len(out) >= limit:
                break
    return out

def list_local_images(dirs: List[Path], limit: int | None = None) -> List[str]:
    files = []
    for d in dirs:
        if not Path(d).exists(): continue
        for ext in ["*.jpg","*.jpeg","*.png","*.webp"]:
            files += list(Path(d).rglob(ext))
    files = [str(p) for p in files]
    return files[:limit] if limit is not None else files

def load_image_any(path: str) -> Image.Image:
    if path.startswith("gs://"):
        _ensure_gcs()
        no_scheme = path[5:]
        with _gcs_fs.open(no_scheme, 'rb') as f:
            data = f.read()
        return Image.open(BytesIO(data)).convert("RGB")
    return Image.open(path).convert("RGB")

# عرض أسرع عبر كاش + تصغير
_img_cache: Dict[str, Image.Image] = {}
def load_image_cached(path: str, max_side=720) -> Image.Image:
    im = _img_cache.get(path)
    if im is not None:
        return im
    im = load_image_any(path)
    w, h = im.size
    s = min(max_side / w, max_side / h, 1.0)
    if s < 1.0:
        im = im.resize((int(w * s), int(h * s)))
    _img_cache[path] = im
    return im

# ==============================================
# Discover dataset
# ==============================================
random.seed(SEED); np.random.seed(SEED)
paths = list_gcs_images(GCS_BUCKET, GCS_PREFIX, limit=None) if USE_GCS else list_local_images(LOCAL_DIRS, limit=None)
random.shuffle(paths)
paths = paths[:MAX_IMAGES] if MAX_IMAGES is not None and len(paths) > MAX_IMAGES else paths
print(f"Device: {device} | Found images: {len(paths)}")
assert len(paths) > 0, "No images found. Check bucket/prefix or local dirs."

# ==============================================
# Light quality + duplicates scan
# ==============================================
def _edge_energy(im: Image.Image) -> float:
    g = im.convert("L").resize((128,128), Image.BILINEAR)
    arr = np.asarray(g, dtype=np.float32)
    dx = np.abs(np.diff(arr, axis=1)).mean()
    dy = np.abs(np.diff(arr, axis=0)).mean()
    return float(dx + dy)

def _ahash(im: Image.Image, size=8) -> int:
    g = im.convert("L").resize((size, size), Image.BILINEAR)
    arr = np.asarray(g, dtype=np.float32)
    thr = arr.mean()
    bits = (arr > thr).astype(np.uint8).flatten()
    h = 0
    for b in bits: h = (h << 1) | int(b)
    return int(h)

@dataclass
class ImageMeta:
    path: str
    width: int
    height: int
    edge_energy: float
    ahash: int

def inspect_images(paths: List[str]) -> List[ImageMeta]:
    out = []
    for p in paths:
        try:
            im = load_image_any(p)
            w, h = im.size
            out.append(ImageMeta(p, w, h, _edge_energy(im), _ahash(im)))
        except Exception:
            continue
    return out

meta = inspect_images(paths)
paths = [m.path for m in meta]
print(f"Inspected: {len(meta)} kept")

hash2idxs: Dict[int, List[int]] = defaultdict(list)
for i, m in enumerate(meta):
    hash2idxs[m.ahash].append(i)
dupe_groups = [v for v in hash2idxs.values() if len(v) > 1]

ahash_groups = {i: set() for i in range(len(paths))}
for group in dupe_groups:
    s = set(group)
    for i in group:
        ahash_groups[i] = s

# ==============================================
# Batch CLIP embeddings + caching
# ==============================================
EMB_DIR = Path("./emb_cache"); EMB_DIR.mkdir(exist_ok=True)
E_PATH = EMB_DIR / "E_fp32.npy"
P_PATH = EMB_DIR / "paths.npy"
M_PATH = EMB_DIR / "meta.json"

FINGERPRINT = {
    "model": MODEL_ID,
    "use_gcs": USE_GCS,
    "bucket": GCS_BUCKET if USE_GCS else None,
    "prefix": GCS_PREFIX if USE_GCS else None,
    "count": len(paths),
}

def get_image_features_batch(batch_imgs: List[Image.Image]) -> np.ndarray:
    inputs = processor(images=batch_imgs, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        emb = model.get_image_features(**inputs).float()
    emb = emb.cpu().numpy()
    emb = l2_normalize(emb, axis=1).astype(np.float32)
    return emb

def encode_all(paths: List[str], batch_size=32) -> np.ndarray:
    E_list, batch = [], []
    for i, p in enumerate(paths, 1):
        try:
            batch.append(load_image_any(p))
        except Exception:
            batch.append(Image.new("RGB", (224,224), (0,0,0)))
        if len(batch) == batch_size or i == len(paths):
            E_list.append(get_image_features_batch(batch)); batch.clear()
        if i % 200 == 0:
            print(f"encoded {i}/{len(paths)}")
    E = np.vstack(E_list).astype(np.float32)
    E = l2_normalize(E, axis=1)
    return E

def save_embeddings(E: np.ndarray, paths: List[str]):
    np.save(E_PATH, E)
    np.save(P_PATH, np.array(paths))
    M_PATH.write_text(json.dumps(FINGERPRINT, ensure_ascii=False))
    try:
        _upload_to_gcs(str(E_PATH)); _upload_to_gcs(str(P_PATH)); _upload_to_gcs(str(M_PATH))
    except Exception:
        pass

def load_embeddings() -> Tuple[np.ndarray | None, List[str] | None, bool]:
    if E_PATH.exists() and P_PATH.exists() and M_PATH.exists():
        try: meta_f = json.loads(M_PATH.read_text())
        except Exception: meta_f = {}
        if meta_f.get("model") == FINGERPRINT["model"] and meta_f.get("use_gcs") == FINGERPRINT["use_gcs"]:
            E = np.load(E_PATH)
            p = np.load(P_PATH, allow_pickle=True).tolist()
            if len(p) == len(paths):
                return E, p, True
    return None, None, False

E, paths2, loaded = load_embeddings()
if loaded and paths2 == paths:
    print("Loaded embeddings from cache.")
else:
    print("Computing embeddings from scratch...")
    E = encode_all(paths, batch_size=32)
    save_embeddings(E, paths)
    print("Saved embeddings to ./emb_cache/")

idx_map = {m.path: i for i, m in enumerate(meta)}
meta = [meta[idx_map[p]] for p in paths]
E = l2_normalize(E, axis=1).astype(np.float32)
N, D = E.shape
print("E shape:", E.shape)

# ==============================================
# Negative prompts + Quality Filter v2 + Quality score
# ==============================================
EMB_DIR.mkdir(exist_ok=True)
QF_META = EMB_DIR / "quality_meta_v2.npz"
NEG_EMB = EMB_DIR / "neg_txt_emb.npy"
NEG_LABS= EMB_DIR / "neg_txt_labels.json"

NEG_PROMPTS = [
    "isolated object on white background","isolated animal on white background",
    "product cutout on white background","plain solid background",
    "macro close-up crop","extreme zoom crop","texture pattern close-up",
    "blurry out of focus photo","logo or icon or clipart",
    "screenshot of user interface","meme with big text","watermark text overlay"
]

def _build_or_load_neg_text_emb():
    if NEG_EMB.exists() and NEG_LABS.exists():
        txt = np.load(NEG_EMB); labs = json.loads(NEG_LABS.read_text())
        if labs == NEG_PROMPTS: return txt
    rows = []
    with torch.no_grad():
        for lab in NEG_PROMPTS:
            prompts = [f"a photo of {lab}", f"high quality {lab}", f"{lab}"]
            tin = processor(text=prompts, return_tensors="pt", padding=True).to(device)
            tfeat = model.get_text_features(**tin).float().cpu().numpy().mean(axis=0)
            rows.append(tfeat)
    neg_txt_emb = l2_normalize(np.stack(rows, axis=0), axis=1).astype(np.float32)
    np.save(NEG_EMB, neg_txt_emb); NEG_LABS.write_text(json.dumps(NEG_PROMPTS))
    try: _upload_to_gcs(str(NEG_EMB)); _upload_to_gcs(str(NEG_LABS))
    except Exception: pass
    return neg_txt_emb

_neg_txt_emb = _build_or_load_neg_text_emb()

def _quick_features_for_path(p: str, thumb=128):
    try:
        im = load_image_any(p)
        has_alpha = (im.mode in ("LA","RGBA","PA"))
        alpha_cut = False
        if has_alpha:
            im = im.convert("RGBA")
            a = np.asarray(im)[:,:,3].astype(np.float32)/255.0
            alpha_cut = (a < 0.05).mean() > 0.10
            im = im.convert("RGB")
        else:
            im = im.convert("RGB")
        w, h = im.size
        im2 = im.copy(); im2.thumbnail((thumb, thumb))
        arr = np.asarray(im2).astype(np.float32)/255.0
        gray = (0.299*arr[:,:,0] + 0.587*arr[:,:,1] + 0.114*arr[:,:,2])
        gx = np.abs(np.diff(gray, axis=1, prepend=gray[:,[0]]))
        gy = np.abs(np.diff(gray, axis=0, prepend=gray[[0],:]))
        edge = gx + gy
        edge_energy = float(edge.mean()*100.0)
        H,W = gray.shape; m1,m2 = int(H*0.25), int(H*0.75); n1,n2 = int(W*0.25), int(W*0.75)
        center = edge[m1:m2, n1:n2].mean() + 1e-8
        border = np.concatenate([edge[:m1,:], edge[m2:,:], edge[m1:m2,:n1], edge[m1:m2,n2:]], axis=None).mean() + 1e-8
        center_edge_ratio = float(center / border)
        std = arr.std(axis=2); white = (arr.mean(axis=2) > 0.92) & (std < 0.03)
        white_bg_frac = float(white.mean())
        return w, h, edge_energy, center_edge_ratio, white_bg_frac, alpha_cut
    except Exception:
        return 0,0,0.0, 1.0, 0.0, False

def _build_or_load_qf_meta(paths: List[str]):
    if QF_META.exists():
        dat = np.load(QF_META, allow_pickle=False)
        if int(dat["N"]) == len(paths): return {k: dat[k] for k in dat.files}
    feats = [_quick_features_for_path(p) for p in paths]
    w,h,ee,cer,wbg,ac = map(np.array, zip(*feats))
    np.savez_compressed(QF_META, N=len(paths), w=w, h=h, edge=ee, center_ratio=cer, whitebg=wbg, alpha_cut=ac)
    try: _upload_to_gcs(str(QF_META))
    except Exception: pass
    return {"N":len(paths), "w":w, "h":h, "edge":ee, "center_ratio":cer, "whitebg":wbg, "alpha_cut":ac}

_qf = _build_or_load_qf_meta(paths)

def compute_quality_mask_v2(min_edge=2.0, min_w=256, min_h=256,
                            detect_zoom=True,  zoom_center_ratio=1.8,
                            detect_cutout=True, solid_bg_frac=0.60, alpha_frac=0.10,
                            use_negative_semantics=True, weird_thresh=0.32):
    w = _qf["w"]; h=_qf["h"]; edge=_qf["edge"]; cer=_qf["center_ratio"]; wbg=_qf["whitebg"]; ac=_qf["alpha_cut"].astype(np.float32)
    ok = (w>=min_w) & (h>=min_h) & (edge>=min_edge)
    if detect_zoom: ok &= (cer <= zoom_center_ratio)
    if detect_cutout: ok &= ~((wbg >= solid_bg_frac) | (ac >= alpha_frac))
    if use_negative_semantics:
        sims = E @ _build_or_load_neg_text_emb().T
        worst = sims.max(axis=1)
        ok &= (worst < weird_thresh)
    return ok.astype(bool)

# --- Quality score ~ edge + megapixels  (0..1)
_MP = (_qf["w"] * _qf["h"]).astype(np.float32) / 1e6
def _minmax01(x):
    lo, hi = np.percentile(x, 5), np.percentile(x, 95)
    return np.clip((x - lo) / (hi - lo + 1e-9), 0.0, 1.0)
Q_EDGE = _minmax01(_qf["edge"])
Q_MP   = _minmax01(_MP)
Q_NORM = 0.5 * Q_EDGE + 0.5 * Q_MP   # 0..1

# ==============================================
# Places-only filter (NEW)
# ==============================================
PL_POS = [
    "wide landscape view","mountain landscape","desert landscape","beach coastline",
    "forest landscape","waterfall in nature","aerial cityscape","city skyline","old town street",
    "architecture exterior","modern building exterior","bridge over river",
    "interior of hotel lobby","modern interior of living room","luxury resort exterior",
    "street scene with buildings","harbor with boats","park landscape","temple exterior","castle exterior"
]
PL_NEG = [
    "portrait of a person","selfie close-up face","close-up of object","macro product",
    "bokeh portrait","face with blurry background","food close-up","hand holding phone",
    "animal close-up face","extreme close-up crop"
]
PL_EMB = EMB_DIR / "places_pos_neg.npz"
def _build_or_load_place_emb():
    if PL_EMB.exists():
        dat = np.load(PL_EMB)
        return dat["pos"], dat["neg"]
    rows_pos, rows_neg = [], []
    with torch.no_grad():
        for lab in PL_POS:
            tin = processor(text=[lab, f"a photo of {lab}", f"high quality {lab}"], return_tensors="pt", padding=True).to(device)
            rows_pos.append(model.get_text_features(**tin).float().cpu().numpy().mean(axis=0))
        for lab in PL_NEG:
            tin = processor(text=[lab, f"a photo of {lab}", f"high quality {lab}"], return_tensors="pt", padding=True).to(device)
            rows_neg.append(model.get_text_features(**tin).float().cpu().numpy().mean(axis=0))
    pos = l2_normalize(np.stack(rows_pos), axis=1).astype(np.float32)
    neg = l2_normalize(np.stack(rows_neg), axis=1).astype(np.float32)
    np.savez_compressed(PL_EMB, pos=pos, neg=neg)
    try: _upload_to_gcs(str(PL_EMB))
    except Exception: pass
    return pos, neg

pl_pos_emb, pl_neg_emb = _build_or_load_place_emb()
def place_scores(E_unit: np.ndarray) -> np.ndarray:
    s_pos = E_unit @ pl_pos_emb.T
    s_neg = E_unit @ pl_neg_emb.T
    return s_pos.max(axis=1) - s_neg.max(axis=1)

PL_SCORE = place_scores(E)

# ==============================================
# TAGS (explain-only, 100 labels)
# ==============================================
label_vocab = [
    "supercars","sports cars","classic cars","luxury cars","off-road trucks",
    "motorcycles","motorsport","yacht","private jet","racing car",
    "luxury watches","mechanical watch","chronograph","jewelry","diamonds",
    "rings","necklaces","handbags","sneakers","menswear","womenswear","streetwear",
    "portraits","studio portrait","headshot","close-up face","street photography","fashion photography",
    "architecture","modern interiors","luxury interiors","minimal interiors",
    "skyscrapers","luxury hotel","resort","spa",
    "landscapes","mountains","beaches","desert","forests","waterfalls",
    "sunsets","night sky","milky way","aurora","aerial",
    "wildlife","dogs","cats","birds","horses","lions","tigers","elephants","wolves","foxes",
    "cityscapes","old town","alleyway","night city","street market",
    "train station","airport lounge","harbor","bridge","tower",
    "food","desserts","coffee","latte art","tea","sushi","pizza","burgers",
    "steak","pasta","salad","breakfast","fine dining",
    "football","basketball","tennis","boxing","golf","gym fitness",
    "cycling","running","swimming","skiing",
    "technology","gadgets","smartphones","laptops","gaming setup","workstation","headphones","camera gear",
    "abstract art","minimalism"
]
assert len(label_vocab) == 100
TEMPLATES = ["a photo of {}", "a high quality photo of {}", "aesthetic {}", "premium {}", "close-up of {}", "{}"]
TXT_EMB_PATH = EMB_DIR / "txt_emb_tags100.npy"
TXT_LABS_PATH= EMB_DIR / "txt_labels_tags100.json"

def build_or_load_text_embeddings():
    if TXT_EMB_PATH.exists() and TXT_LABS_PATH.exists():
        txt_emb = np.load(TXT_EMB_PATH)
        labs = json.loads(TXT_LABS_PATH.read_text())
        if labs == label_vocab and txt_emb.shape[0] == len(label_vocab):
            return txt_emb
    with torch.no_grad():
        rows = []
        for lab in label_vocab:
            prompts = [t.format(lab) for t in TEMPLATES]
            tin = processor(text=prompts, return_tensors="pt", padding=True).to(device)
            tfeat = model.get_text_features(**tin).float().cpu().numpy().mean(axis=0)
            rows.append(tfeat)
    txt_emb = l2_normalize(np.stack(rows, axis=0), axis=1).astype(np.float32)
    np.save(TXT_EMB_PATH, txt_emb); TXT_LABS_PATH.write_text(json.dumps(label_vocab, ensure_ascii=False))
    try: _upload_to_gcs(str(TXT_EMB_PATH)); _upload_to_gcs(str(TXT_LABS_PATH))
    except Exception: pass
    return txt_emb

txt_emb = build_or_load_text_embeddings()

def image_labels_from_idx(E_unit: np.ndarray, idx: int, top_k: int = 6):
    e = E_unit[idx]; scores = (txt_emb @ e); J = np.argsort(-scores)[:top_k]
    return [(label_vocab[j], float(scores[j])) for j in J]

def orientation_from_pref(pref: np.ndarray, top_k: int = 8):
    scores = (txt_emb @ pref); J = np.argsort(-scores)[:top_k]
    return [(label_vocab[j], float(scores[j])) for j in J]

# ==============================================
# Recommender with dynamic η + smart selection
# ==============================================
class ImageRecommender:
    def __init__(self, E_unit: np.ndarray, quality_mask: np.ndarray | None = None,
                 alpha=0.85, eta=1.6, warmup_n=5,
                 recent_k=50, recent_weight=0.8, focus_gamma=1.3,
                 diversity_last_k=20, diversity_min_cos=0.92,
                 hide_exact_dupes=True, rng=None):
        self.E = l2_normalize(np.asarray(E_unit), axis=1)
        self.N, self.D = self.E.shape

        self.alpha = float(alpha); self.eta = float(eta)
        self.eta0 = float(eta);    # init LR
        self.eta_min = 0.10
        self.eta_decay_span = 50
        self.use_decay = True      # toggle from UI

        self.warmup_n = int(warmup_n); self.focus_gamma = float(focus_gamma)
        self.recent_k = int(recent_k); self.recent_weight = float(recent_weight)
        self.diversity_last_k = int(diversity_last_k); self.diversity_min_cos = float(diversity_min_cos)
        self.hide_exact_dupes = bool(hide_exact_dupes)

        self.rng = rng or np.random.default_rng(42)
        base = self.E.mean(axis=0) + self.rng.normal(0.0, 0.05, size=self.D)
        self.preference = l2_normalize(base).astype(np.float32)

        self.quality_mask = quality_mask if quality_mask is not None else np.ones(self.N, bool)
        self.seen = set()
        self._last_shown = deque(maxlen=self.diversity_last_k)
        self._recent = deque(maxlen=self.recent_k)

        self._warm_count = 0; self._warm_sum = np.zeros(self.D, np.float32)
        self._warm_weight = 0.0; self._warmed_up = (self.warmup_n <= 0)
        self._warm_order = self.rng.permutation(self.N); self._warm_ptr = 0
        self._updates_total = 0
        self._eta_updates = 0

    def set_quality_mask(self, mask: np.ndarray):
        self.quality_mask = mask

    def set_diversity(self, last_k: int, min_cos: float):
        self.diversity_last_k = int(last_k); self.diversity_min_cos = float(min_cos)
        self._last_shown = deque(list(self._last_shown), maxlen=self.diversity_last_k)

    def _passes_diversity(self, idx: int) -> bool:
        if not self._last_shown or self.diversity_min_cos >= 0.999: return True
        e = self.E[idx]
        return all(float(np.dot(e, self.E[li])) < self.diversity_min_cos for li in self._last_shown)

    def _mask_candidates(self) -> np.ndarray:
        mask = self.quality_mask.copy()
        if self.seen: mask[list(self.seen)] = False
        if self.hide_exact_dupes and self.seen:
            banned = set()
            for i in self.seen: banned |= ahash_groups.get(i, set())
            if banned: mask[list(banned)] = False
        return mask

    def _pick_warmup(self):
        mask = self._mask_candidates()
        while self._warm_ptr < self.N:
            idx = int(self._warm_order[self._warm_ptr]); self._warm_ptr += 1
            if not mask[idx] or not self._passes_diversity(idx): continue
            return idx
        return None

    def _update(self, idx: int, feedback: float):
        e = self.E[idx]; self.seen.add(idx); self._last_shown.append(idx)
        self._updates_total += 1

        # warm-up phase
        if not self._warmed_up:
            self._warm_count += 1
            if feedback > 0:
                self._warm_sum += feedback * e
                self._warm_weight += feedback
            if (self._warm_count >= self.warmup_n) or (self._warm_weight >= 1.0):
                if self._warm_weight > 0:
                    delta = self._warm_sum / max(self._warm_weight, 1e-9)
                    self.preference = l2_normalize(self.alpha * self.preference + self.eta * delta)
                self._warmed_up = True; self._warm_sum[:] = 0; self._warm_weight = 0.0
            return

        # dynamic η (optional decay to floor)
        if self.use_decay:
            self._eta_updates += 1
            t = min(self._eta_updates / max(1.0, float(self.eta_decay_span)), 1.0)
            self.eta = self.eta_min + (self.eta0 - self.eta_min) * (1.0 - t)
        else:
            self.eta = self.eta0

        # online EMA
        if feedback > 0:
            self._recent.append(feedback * e)
        recent_centroid = (np.mean(np.stack(self._recent, axis=0), axis=0)
                           if (self.recent_weight > 0 and len(self._recent) > 0) else 0.0)
        update_vec = self.alpha * self.preference + self.eta * feedback * e + self.recent_weight * recent_centroid
        self.preference = l2_normalize(update_vec)

    # اختيار ذكي: تفضيل + تنوّع + جودة + منع near-dup
    def recommend_next_smart(self, pool_k=300, lambda_div=0.50, quality_boost=0.25, near_dupe_thr=0.95):
        if len(self.seen) >= self.N: return None, None
        if not self._warmed_up:
            idx = self._pick_warmup(); return (idx, None) if idx is not None else (None, None)
        sims = self.E @ self.preference
        mask = self._mask_candidates()
        sims = np.where(mask, sims, -np.inf)
        k = min(pool_k, np.isfinite(sims).sum())
        if k <= 0: return None, None
        pool = np.argpartition(-sims, range(k))[:k]

        # near-duplicate barrier
        if len(self._last_shown):
            sim_to_sel = self.E[pool] @ self.E[list(self._last_shown)].T
            too_close = (sim_to_sel.max(axis=1) >= float(near_dupe_thr))
            pool = pool[~too_close]
            if pool.size == 0:
                pool = np.argpartition(-sims, range(k))[:k]

        if not len(self._last_shown):
            j = int(pool[np.argmax(sims[pool])]); return j, float(sims[j])

        sim_to_sel = self.E[pool] @ self.E[list(self._last_shown)].T
        novelty = 1.0 - sim_to_sel.max(axis=1)        # higher = more different
        q = Q_NORM[pool].astype(np.float32)           # quality (0..1)
        pref_part = sims[pool]

        score = (1.0 - lambda_div) * pref_part + lambda_div * novelty + float(quality_boost) * q
        j_local = int(np.argmax(score)); j = int(pool[j_local])
        return j, float(sims[j])

# ==============================================
# Instantiate
# ==============================================
quality_mask = compute_quality_mask_v2(2.0, 256, 256) & (place_scores(E) >= 0.05)  # default: places gate on
rec = ImageRecommender(
    E, quality_mask=quality_mask,
    alpha=0.85, eta=1.6, warmup_n=5,
    recent_k=50, recent_weight=0.8,
    focus_gamma=1.3, diversity_last_k=20, diversity_min_cos=0.92,
    hide_exact_dupes=True
)
print("Recommender ready:", rec.N, "items; D=", rec.D)

# ==============================================
# UI + User persistence (Accumulated)
# ==============================================
# Quality controls
min_edge = W.FloatSlider(value=2.0, min=0.0, max=10.0, step=0.1, description='min_edge')
min_w    = W.IntSlider(value=256, min=128, max=2048, step=64, description='min_w')
min_h    = W.IntSlider(value=256, min=128, max=2048, step=64, description='min_h')
detect_zoom = W.Checkbox(value=True, description='detect_zoom')
zoom_center_ratio = W.FloatSlider(value=1.8, min=1.0, max=3.0, step=0.05, description='zoom_ratio')
detect_cutout = W.Checkbox(value=True, description='detect_cutout')
solid_bg_frac = W.FloatSlider(value=0.60, min=0.0, max=1.0, step=0.05, description='solid_bg')
alpha_frac    = W.FloatSlider(value=0.10, min=0.0, max=1.0, step=0.05, description='alpha_frac')
use_neg = W.Checkbox(value=True, description='neg_semantics')
weird_thresh = W.FloatSlider(value=0.32, min=0.1, max=0.7, step=0.02, description='neg_thresh')

# Places-only
places_only = W.Checkbox(value=True, description='places_only')
place_min   = W.FloatSlider(value=0.05, min=-0.5, max=0.5, step=0.01, description='place_min')
places_box = W.VBox([W.HTML("<b>Places-only filter</b>"), W.HBox([places_only, place_min])])

quality_box = W.VBox([
    W.HTML("<b>Quality filters</b>"),
    W.HBox([min_edge, min_w, min_h]),
    W.HBox([detect_zoom, zoom_center_ratio]),
    W.HBox([detect_cutout, solid_bg_frac, alpha_frac]),
    W.HBox([use_neg, weird_thresh]),
    places_box
])

# Model params
alpha_s       = W.FloatSlider(value=0.85, min=0.50, max=0.99, step=0.01, description='alpha')
recent_weight = W.FloatSlider(value=0.80, min=0.00, max=1.50, step=0.05, description='recent_w')
focus_gamma   = W.FloatSlider(value=1.30, min=1.00, max=2.00, step=0.01, description='focus_γ')

model_box = W.VBox([
    W.HTML("<b>Model params</b>"),
    W.HBox([alpha_s, recent_weight, focus_gamma])
])


# Learning rate (with decay toggle)
lr_init   = W.FloatSlider(value=1.60, min=0.05, max=3.00, step=0.05, description='η_init')
lr_min    = W.FloatSlider(value=0.10, min=0.00, max=1.00, step=0.01, description='η_min')
lr_span   = W.IntSlider(value=50, min=1, max=500, step=1, description='η_span')
use_decay = W.Checkbox(value=True, description='use_decay')
lr_now    = W.Label(value="η_now=—")
lr_box = W.VBox([W.HTML("<b>Learning rate</b>"), W.HBox([lr_init, lr_min, lr_span, use_decay]), lr_now])

# Diversity & Quality (selection)
lambda_div    = W.FloatSlider(value=0.50, min=0.0, max=1.0, step=0.05, description='λ_div')
near_dupe_thr = W.FloatSlider(value=0.95, min=0.80, max=0.999, step=0.005, description='near_dupe')
quality_boost = W.FloatSlider(value=0.30, min=0.0, max=1.0, step=0.05, description='Q_boost')
pool_k        = W.IntSlider(value=300, min=50, max=1000, step=10, description='pool_k')
smart_box = W.VBox([W.HTML("<b>Diversity & Quality</b>"), W.HBox([lambda_div, near_dupe_thr, quality_boost, pool_k])])

# Warm-up controls (sample size editable)
warmup_n       = W.IntSlider(value=5, min=0, max=50, step=1, description='warmup_n')
diverse_warmup = W.Checkbox(value=True, description='diverse warmup')
build_warmup   = W.Button(description="Build warmup…")
warmup_box = W.VBox([W.HTML("<b>Warm-up (first images)</b>"), W.HBox([warmup_n, diverse_warmup, build_warmup])])

# User profile (load/save accum)
user_id        = W.Text(value="default", description='user_id')
btn_user_load  = W.Button(description="Load user")
btn_user_save  = W.Button(description="Save user")
user_box       = W.VBox([W.HTML("<b>User profile</b>"), W.HBox([user_id, btn_user_load, btn_user_save])])

# Debug: vectors/matrices (lightweight text views)
k_head = W.IntSlider(value=16, min=8, max=64, step=2, description='head_k')
pref_area  = W.Textarea(value="", layout=W.Layout(width='600px', height='150px'), disabled=True)
img_area   = W.Textarea(value="", layout=W.Layout(width='600px', height='150px'), disabled=True)
accum_area = W.Textarea(value="", layout=W.Layout(width='600px', height='150px'), disabled=True)
btn_refresh_vec = W.Button(description="Refresh vectors")
vec_box = W.VBox([
    W.HTML("<b>Vectors (debug)</b>"),
    k_head,
    W.HBox([W.VBox([W.HTML("<b>Preference vector</b>"), pref_area]),
            W.VBox([W.HTML("<b>Current image vector</b>"), img_area])]),
    W.VBox([W.HTML("<b>User accum (mean)</b>"), accum_area]),
    btn_refresh_vec
])

# Top-level buttons
btn_apply = W.Button(description="Apply filters & params", button_style='info')
btn_next  = W.Button(description="Next ▶")
btn_like  = W.Button(description="Like 🔥", button_style='success')
btn_reset = W.Button(description="Reset (to accum)", button_style='warning')
buttons   = W.HBox([btn_apply, btn_next, btn_like, btn_reset])

# Output area for image & logs
out = W.Output(layout={'border': '1px solid #ddd', 'padding': '6px'})

# ---------- Persistence helpers ----------
STATE_DIR = Path("./session_state"); STATE_DIR.mkdir(exist_ok=True)
def _load_user_db():
    if USER_DB_PATH.exists():
        try: return json.loads(USER_DB_PATH.read_text())
        except Exception: return {}
    return {}
def _save_user_db(db: dict):
    USER_DB_PATH.parent.mkdir(exist_ok=True)
    USER_DB_PATH.write_text(json.dumps(db))
    try: _upload_to_gcs(str(USER_DB_PATH))
    except Exception: pass

_user_accum_sum = np.zeros(D, np.float32)
_user_accum_w   = 0.0

def _user_load(_=None):
    global _user_accum_sum, _user_accum_w
    db = _load_user_db(); u = user_id.value.strip() or 'default'
    ent = db.get(u, {})
    s = np.array(ent.get('accum_sum', np.zeros(D).tolist()), dtype=np.float32)
    w = float(ent.get('accum_weight', 0.0))
    if s.shape[0] != D: s = np.zeros(D, np.float32); w = 0.0
    _user_accum_sum = s; _user_accum_w = w
    if w > 0: rec.preference = l2_normalize(s / max(w, 1e-9))
    else:
        base = rec.E.mean(axis=0) + np.random.default_rng(42).normal(0.0, 0.05, size=rec.D)
        rec.preference = l2_normalize(base).astype(np.float32)
    with out:
        clear_output(wait=True); print(f"Loaded user '{u}'. Accum weight={_user_accum_w:.2f}")

def _user_save(_=None):
    db = _load_user_db(); u = user_id.value.strip() or 'default'
    db[u] = {'accum_sum': _user_accum_sum.tolist(), 'accum_weight': _user_accum_w, 'updated': time.time()}
    _save_user_db(db)
    with out:
        clear_output(wait=True); print(f"Saved user '{u}'. Accum weight={_user_accum_w:.2f}")

btn_user_load.on_click(_user_load)
btn_user_save.on_click(_user_save)

def _commit_user_accum(idx: int, feedback: float):
    # Only positive feedback contributes to long-term accum
    global _user_accum_sum, _user_accum_w
    if feedback <= 0: return
    e = rec.E[idx]
    _user_accum_sum = (_user_accum_sum + feedback * e).astype(np.float32)
    _user_accum_w   = float(_user_accum_w + feedback)
    _user_save()

# ---------- Runtime helpers ----------
from time import perf_counter
_last_t0 = None
_current_idx = None

def _apply_params_to_model():
    rec.alpha         = float(alpha_s.value)
    rec.recent_weight = float(recent_weight.value)
    rec.focus_gamma   = float(focus_gamma.value)
    rec.warmup_n      = int(warmup_n.value)
    rec.eta0          = float(lr_init.value)
    rec.eta_min       = float(lr_min.value)
    rec.eta_decay_span= int(lr_span.value)
    rec.use_decay     = bool(use_decay.value)
    rec.eta           = rec.eta0
    lr_now.value = f"η_now={rec.eta:.3f}"

def _recompute_quality():
    q = compute_quality_mask_v2(
        min_edge=float(min_edge.value),
        min_w=int(min_w.value), min_h=int(min_h.value),
        detect_zoom=bool(detect_zoom.value), zoom_center_ratio=float(zoom_center_ratio.value),
        detect_cutout=bool(detect_cutout.value), solid_bg_frac=float(solid_bg_frac.value), alpha_frac=float(alpha_frac.value),
        use_negative_semantics=bool(use_neg.value), weird_thresh=float(weird_thresh.value),
    )
    if bool(places_only.value):
        q &= (PL_SCORE >= float(place_min.value))
    rec.set_quality_mask(q)
    return int(q.sum())

def _build_diverse_warmup(k: int):
    mask = rec._mask_candidates()
    I = np.where(mask)[0]
    if I.size == 0: return
    POOL = I if I.size <= 1500 else np.random.default_rng(0).choice(I, size=1500, replace=False)
    Epool = rec.E[POOL]
    rng = np.random.default_rng(0)
    sel = [int(rng.integers(low=0, high=len(POOL)))]
    while len(sel) < min(k, len(POOL)):
        sims = Epool @ Epool[sel].T
        max_sim = sims.max(axis=1)
        cand = int(np.argmin(max_sim))
        if cand in sel: break
        sel.append(cand)
    warm = POOL[sel].tolist()
    rest = [int(x) for x in POOL if int(x) not in warm]
    rng.shuffle(rest)
    order = warm + rest + [int(x) for x in I if int(x) not in set(POOL)]
    rec._warm_order = np.array(order, dtype=np.int64); rec._warm_ptr = 0

def _fmt_vec(v: np.ndarray, k: int):
    v = np.asarray(v, dtype=np.float32).reshape(-1)
    head = ", ".join([f"{x:.3f}" for x in v[:k]])
    return f"shape=({v.size},)  norm={float(np.linalg.norm(v)):.3f}\nhead[{k}]: [{head}]"

def _update_debug_views(idx: int | None):
    # Preference
    pref_area.value = _fmt_vec(rec.preference, int(k_head.value))
    # Current image vector
    if idx is not None:
        img_area.value = _fmt_vec(rec.E[idx], int(k_head.value))
    else:
        img_area.value = "(no image yet)"
    # Accum mean
    if _user_accum_w > 0:
        mean_v = _user_accum_sum / max(_user_accum_w, 1e-9)
        accum_area.value = _fmt_vec(mean_v, int(k_head.value)) + f"\nweight={_user_accum_w:.3f}"
    else:
        accum_area.value = "(empty accum)"

def _render(idx, note=""):
    global _last_t0, _current_idx
    _current_idx = idx; _last_t0 = perf_counter()
    p = paths[idx]; sim = float(np.dot(rec.E[idx], rec.preference))
    labels = image_labels_from_idx(rec.E, idx, top_k=6)
    orient = orientation_from_pref(rec.preference, top_k=6)
    im = load_image_cached(p, max_side=720)
    with out:
        clear_output(wait=True)
        display(im)
        print(f"[ONLINE] idx={idx} | sim={sim:.3f} | seen={len(rec.seen)} / {rec.N} | {note} | η_now={rec.eta:.3f}")
        print("Image labels:", ", ".join([t for t,_ in labels]))
        print("User orientation:", ", ".join([t for t,_ in orient]))
        print(f"Accum weight={_user_accum_w:.2f}")
    _update_debug_views(idx)

def _compute_feedback(action: str, dwell_seconds: float | None):
    base = LIKE_BASE if action == 'like' else SWIPE_FAST
    # dwell bonus for swipes only (NOT likes) per your request
    if action != 'like' and dwell_seconds is not None and dwell_seconds >= float(DWELL_THRESHOLD):
        base += DWELL_BONUS
    return float(base)

def _next(apply_prev_feedback: bool = True):
    if apply_prev_feedback and _current_idx is not None:
        dwell = perf_counter() - _last_t0 if _last_t0 is not None else 0.0
        fb = _compute_feedback('swipe', dwell)
        _commit_user_accum(int(_current_idx), fb)
        sim_before = float(np.dot(rec.E[int(_current_idx)], rec.preference))
        rec._update(int(_current_idx), float(fb))
        lr_now.value = f"η_now={rec.eta:.3f}"
        sim_after = float(np.dot(rec.E[int(_current_idx)], rec.preference))
        note = f"SWIPE: dwell={dwell:.2f}s → fb={fb:+.2f} | sim {sim_before:.3f}→{sim_after:.3f}"
    else:
        note = "NEXT"

    idx, _ = rec.recommend_next_smart(
        pool_k=int(pool_k.value),
        lambda_div=float(lambda_div.value),
        quality_boost=float(quality_boost.value),
        near_dupe_thr=float(near_dupe_thr.value)
    )
    if idx is None:
        with out:
            clear_output(wait=True); print("No candidate found. Relax filters or press Reset.")
        _update_debug_views(None)
        return
    _render(idx, note=note)

def _like():
    if _current_idx is None:
        _next(apply_prev_feedback=False); return
    dwell = perf_counter() - _last_t0 if _last_t0 is not None else None
    fb = _compute_feedback('like', dwell)  # like ignores dwell bonus
    _commit_user_accum(int(_current_idx), fb)
    sim_before = float(np.dot(rec.E[int(_current_idx)], rec.preference))
    rec._update(int(_current_idx), float(fb))
    lr_now.value = f"η_now={rec.eta:.3f}"
    sim_after = float(np.dot(rec.E[int(_current_idx)], rec.preference))
    note = f"LIKE: fb={fb:+.2f} | sim {sim_before:.3f}→{sim_after:.3f}"

    idx, _ = rec.recommend_next_smart(
        pool_k=int(pool_k.value),
        lambda_div=float(lambda_div.value),
        quality_boost=float(quality_boost.value),
        near_dupe_thr=float(near_dupe_thr.value)
    )
    if idx is None:
        with out:
            clear_output(wait=True); print("End of candidates. Try Reset.")
        _update_debug_views(None)
        return
    _render(idx, note=note)

def _reset_user():
    if _user_accum_w > 0:
        rec.preference = l2_normalize(_user_accum_sum / _user_accum_w)
    else:
        base = rec.E.mean(axis=0) + np.random.default_rng(42).normal(0.0, 0.05, size=rec.D)
        rec.preference = l2_normalize(base).astype(np.float32)
    rec.seen.clear(); rec._last_shown.clear(); rec._recent.clear()
    rec._warm_count = 0; rec._warm_sum[:] = 0; rec._warm_weight = 0.0
    rec._warmed_up = (rec.warmup_n <= 0); rec._warm_ptr = 0
    rec._updates_total = 0; rec._eta_updates = 0; rec.eta = rec.eta0
    with out:
        clear_output(wait=True); print("User state reset to ACCUM profile. Press Next to start.")
    _update_debug_views(None)

# Wire buttons
def on_apply_clicked(_):
    kept = _recompute_quality(); _apply_params_to_model()
    with out:
        clear_output(wait=True)
        print(f"Applied. Kept {kept}/{len(rec.quality_mask)} | warmup_n={rec.warmup_n} | η_init={rec.eta0:.2f} "
              f"→ η_min={rec.eta_min:.2f} in {rec.eta_decay_span} steps | use_decay={rec.use_decay}")
btn_apply.on_click(on_apply_clicked)
btn_next.on_click(lambda _: _next(True))
btn_like.on_click(lambda _: _like())
btn_reset.on_click(lambda _: _reset_user())
btn_refresh_vec.on_click(lambda _: _update_debug_views(_current_idx))

def on_build_warmup(_):
    if bool(diverse_warmup.value):
        _build_diverse_warmup(int(warmup_n.value))
    else:
        mask = rec._mask_candidates(); I = np.where(mask)[0]
        order = np.array(np.random.default_rng(0).permutation(I), dtype=np.int64)
        rec._warm_order = order; rec._warm_ptr = 0
    with out:
        clear_output(wait=True); print(f"Warm-up order prepared. diverse={bool(diverse_warmup.value)} | k={int(warmup_n.value)}")
build_warmup.on_click(on_build_warmup)

# Layout & init
ui = W.VBox([
    user_box,
    W.HBox([quality_box, model_box, lr_box]),
    W.HBox([smart_box, warmup_box]),
    vec_box,
    buttons,
    out
])
display(ui)

# Initialize session
_user_load()
_recompute_quality()
_apply_params_to_model()
on_build_warmup(None)

# Show first image
idx, _ = rec.recommend_next_smart(
    pool_k=int(pool_k.value),
    lambda_div=float(lambda_div.value),
    quality_boost=float(quality_boost.value),
    near_dupe_thr=float(near_dupe_thr.value)
)
if idx is not None:
    _render(idx, note="INIT")
else:
    with out:
        clear_output(wait=True); print("No candidate at init. Relax filters.")
    _update_debug_views(None)


Note: you may need to restart the kernel to use updated packages.


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Device: cpu | Found images: 100
Inspected: 100 kept
Loaded embeddings from cache.
E shape: (100, 512)
Recommender ready: 100 items; D= 512
